# 推理

## 任务模式

In [ ]:
# Llama
#!pip install --upgrade transformers -q

import transformers
import torch

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

model_id = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto"
)

from IPython.display import Markdown

messages = [
    {"role": "system", "content": "您扮演位中医药专家"},
    {"role": "user", "content": "板蓝根怎么样"},
]

outputs = pipeline(
    messages,
    max_new_tokens=2048,
)

Markdown(outputs[0]["generated_text"][-1]['content'])

In [ ]:
# DeepSeek
#!pip install --upgrade transformers -q

import transformers
import torch

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

pipeline = transformers.pipeline(
    "text-generation",
    model="/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-llama-8b/1",
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto"
)

from IPython.display import Markdown

messages = [
    {"role": "system", "content": "您扮演位中医药专家"},
    {"role": "user", "content": "板蓝根怎么样"},
]

outputs = pipeline(
    messages,
    max_new_tokens=2048,
)

Markdown(outputs[0]["generated_text"][-1]['content'])

## 生成模式

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained( model_id, device_map='auto')

model

In [ ]:
from IPython.display import Markdown

prompt = "夺岛作战方案"
messages = [
    {"role": "system", "content": "妳作为作战小专家"},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

Markdown(response)

# 微调

In [ ]:
!pip install -q -U bitsandbytes
!pip install -q -U transformers peft accelerate trl

## 加载模型

In [ ]:
import torch
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# Set the data type for computations to float16, bfloat16 not supported on T4/P100
compute_dtype = getattr(torch, "float16")

# Configure the BitsAndBytes settings for 4-bit quantization to reduce memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enable 4-bit quantization
    bnb_4bit_use_double_quant=True,  # Use double quantization for improved precision
    bnb_4bit_quant_type="nf4",  # Specify the quantization type
    bnb_4bit_compute_dtype=compute_dtype,  # Set the computation data type
)

model_id = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"

# Load the pre-trained model with specified configurations
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,  # Apply the 4-bit quantization configuration
    torch_dtype=compute_dtype,  # Set the data type for the model
    use_cache=False,  # Disable caching to save memory
    device_map='auto',  # Automatically map the model to available devices (e.g., GPUs)
)

# Enable gradient checkpointing to reduce memory usage during backpropagation
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model

In [ ]:
# Load the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(
    model_id, device_map=model.device, use_cache=False
    model_max_length=512,
    padding_side="left",
    add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from IPython.display import Markdown

prompt = "我有一个信息科学相关的问题，什么是 zsh？"
messages = [
    {"role": "system", "content": "您是位IT知识智能助手"},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

Markdown(response)

## 数据准备

In [ ]:
# 公开数据集
from datasets import load_dataset
dataset = load_dataset("imdb", split="train")

In [ ]:
# 自定义数据

!wget -O baike.jsonl https://huggingface.co/datasets/Hello-SimpleAI/HC3-Chinese/raw/main/baike.jsonl

import pandas as pd
dataset_path = "baike.jsonl"
train_df = pd.read_json(dataset_path, lines=True)[:100]
train_df[:3]

## 数据加工

In [ ]:
# chatML

def process_func(example):
    """
    将数据集进行预处理
    """
    MAX_LENGTH = 384
    input_ids, attention_mask, labels = [], [], []

    instruction = tokenizer(
        # 加入Qwen特殊符防提示词注入
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n您是位IT知识智能助手<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{example['question']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
        add_special_tokens=False
    )
    
    response = tokenizer(f"{example['human_answers']}", add_special_tokens=False)
    
    input_ids = (
        instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    )
    
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    labels = (
        [-100] * len(instruction["input_ids"])
        + response["input_ids"]
        + [tokenizer.pad_token_id]
    )
    #attention_mask = input_ids != tokenizer.pad_token_id
    
    if len(input_ids) > MAX_LENGTH:  # 做截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
        
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
# 转换成数据集
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_func, remove_columns=train_ds.column_names, num_proc=4)
train_dataset

In [ ]:
tokenizer.decode(train_dataset[0]['input_ids'])

## 低秩优化

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False,  # 训练模式
    r=8,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
)

model_lora = get_peft_model(model, config)

## 原生训练SFT

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./output_Instruct-FT",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=10,
    num_train_epochs=1,
    save_steps=10,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none"
)

## 训练生成

In [ ]:
from transformers import Trainer, DataCollatorForSeq2Seq

torch.cuda.amp.autocast(enabled=True, dtype=torch.float16, cache_enabled=True)

trainer = Trainer(
    model=model_lora,
    args=args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

trainer.train()

# 强化训练RLHF

In [ ]:
!pip install trl -q

In [ ]:
# 0. imports
import torch
from transformers import GPT2Tokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

# 1. load a pretrained model
model = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2", device_map='cuda')
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2", device_map='cuda')
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 2. initialize trainer
ppo_config = {"mini_batch_size": 1, "batch_size": 1, "output_dir": "./"}
config = PPOConfig(**ppo_config)
ppo_trainer = PPOTrainer(config, model, ref_model, tokenizer)

# 3. encode a query
query_txt = "This morning I went to the "
query_tensor = tokenizer.encode(query_txt, return_tensors="pt").to(model.pretrained_model.device)

# 4. generate model response
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 20,
}
response_tensor = ppo_trainer.generate([item for item in query_tensor], return_prompt=False, **generation_kwargs)
response_txt = tokenizer.decode(response_tensor[0])

# 5. define a reward for response
# (this could be any reward such as human feedback or output from another model)
reward = [torch.tensor(1.0, device=model.pretrained_model.device)]

# 6. train model with ppo
train_stats = ppo_trainer.step([query_tensor[0]], [response_tensor[0]], reward)

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer

dataset = load_dataset("imdb", split="train")

trainer = SFTTrainer(
    model=model_lora,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512
)

trainer.train()

In [ ]:
import torch
from transformers import AutoTokenizer
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, create_reference_model
from trl.core import respond_to_batch

# 首先加载模型，然后创建参考模型
model_ref = create_reference_model(model)

model_ref

In [ ]:
# 初始化ppo配置对象
ppo_config = PPOConfig(
    batch_size=1, output_dir='./'
)

# 编码一个query
query_txt = "This morning I went to the "
query_tensor = tokenizer.encode(query_txt, return_tensors="pt")

# 得到模型response
response_tensor  = respond_to_batch(model, query_tensor)

# 创建一个ppo trainer
ppo_trainer = PPOTrainer(ppo_config, policy=model, ref_policy=model_ref, train_dataset=tokenizer)

# 为response定义一个reward（人类反馈或模型输出奖励） 
reward = [torch.tensor(1.0)]

# 使用ppo训练一步模型
train_stats = ppo_trainer.step([query_tensor[0]], [response_tensor[0]], reward)

In [ ]:
## 保存 LoRa 权重和分词
trainer.model.save_pretrained('./model-lora')
tokenizer.save_pretrained('./model-lora')

In [ ]:
from peft import PeftModel

lora_model = PeftModel.from_pretrained(
    model, model_id='/kaggle/working/model-lora',
    #config=config, 
    is_trainable=False
)
lora_model